<a href="https://www.kaggle.com/code/asopozala/bbc-news-building-a-topic-feature-dataset?scriptVersionId=319328316" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

This notebook focuses on constructing a clean, reusable topic–feature dataset from BBC news articles. I transform raw text into numerical representations (e.g., TF–IDF, NMF topics) and package them with labels so they can be used by multiple downstream models and experiments.

In [ ]:
import pandas as pd

data_path = "/kaggle/input/datasets/gpreda/bbc-news/bbc_news.csv"
df = pd.read_csv(data_path)

df["pubDate_dt"] = pd.to_datetime(df["pubDate"], errors="coerce")
df["year"] = df["pubDate_dt"].dt.year

df = df[df["year"].isin([2022, 2023, 2024])].copy()
df["text"] = df["title"].astype(str) + " " + df["description"].astype(str)

df[["year", "title"]].head()


### Step 1 – Build a topic‑feature dataset

Goal: turn the NMF output into a clean table we can reuse in other notebooks.

- Each article gets 17 topic weights (one column per topic).
- We combine these topic features with simple metadata (date, year, title).
- This table will let us explore connections between topics and years, and later export it as a new dataset for predictive modeling.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.8,
    min_df=10
)

tfidf = vectorizer.fit_transform(df["text"])
tfidf.shape


In [ ]:
from sklearn.decomposition import NMF
import numpy as np

n_components = 17
nmf = NMF(n_components=n_components, random_state=0)
nmf_features = nmf.fit_transform(tfidf)

feature_names = vectorizer.get_feature_names_out()
n_top_words = 10

for topic_idx, topic in enumerate(nmf.components_):
    top_indices = np.argsort(topic)[::-1][:n_top_words]
    top_words = [feature_names[i] for i in top_indices]
    print(f"Topic {topic_idx}: {' | '.join(top_words)}")


In [ ]:
topic_cols = [f"topic_{i}" for i in range(n_components)]
topic_df = pd.DataFrame(nmf_features, columns=topic_cols, index=df.index)

features = pd.concat(
    [df[["pubDate_dt", "year", "title"]].reset_index(drop=True),
     topic_df.reset_index(drop=True)],
    axis=1
)

features.head()


### Step 2 – Dominant vs. mixed topic articles

Now I want to see, for each article, whether:
- One topic is **dominant** (for example, more than 50% of its total NMF weight), or
- The article is more **mixed**, with several topics having similar weights.

To do this, I will:
1. Normalise the 17 topic weights per article so they become proportions that sum to 1.
2. Look at the **maximum proportion** per article (how strong the top topic is) and count how many articles have a very dominant topic vs a more even mix.


In [ ]:
import numpy as np

# 1) get topic weights as array
topic_weights = features[topic_cols].values

# 2) normalise per article so topics become proportions that sum to 1
row_sums = topic_weights.sum(axis=1, keepdims=True)
topic_props = topic_weights / row_sums

# 3) store a few summary stats back into the dataframe
features["max_topic_prop"] = topic_props.max(axis=1)
features["n_topics_prop_over_0_1"] = (topic_props >= 0.1).sum(axis=1)

features[["year", "title", "max_topic_prop", "n_topics_prop_over_0_1"]].head()


### Step 3 – Keep only well‑aligned articles (max topic ≥ 0.3)

Idea: some articles do not align strongly with any of the 17 NMF topics; all topic proportions stay small and similar.  
I look at the strongest topic proportion for each article (`max_topic_prop`) and mark articles as **“well aligned”** if their top topic has at least **0.3** of the total weight.  
This lets me separate:  
- articles clearly dominated by one topic (max ≥ 0.3), and  
- more diffuse, hard‑to‑interpret articles (max < 0.3), which I may drop for later predictive modelling.


In [ ]:
threshold = 0.3
features["well_aligned"] = features["max_topic_prop"] >= threshold

print("Articles total:", len(features))
print("Well-aligned (max_topic_prop ≥ 0.3):", features["well_aligned"].sum())
print("Not well-aligned:", (~features["well_aligned"]).sum())

features[["year", "title", "max_topic_prop", "well_aligned"]].head(10)


In [ ]:
# final cleaned topic‑feature dataset
features_clean = features[features["well_aligned"]].copy()

print("Rows in cleaned dataset:", len(features_clean))
features_clean.head()


In [ ]:
cols_to_view = [
    "year", "title",
    "max_topic_prop", "dominant_topic", "strongly_dominant",
    "n_topics_prop_over_0_1", "n_topics_to_cover_0_7"
]

sample_clean = features_clean[cols_to_view].sample(20, random_state=42)
sample_clean


### Step 4 – Export cleaned topic‑feature dataset for reuse

Now I save the cleaned topic‑feature table as a CSV file.  
Later I will upload this CSV as a new Kaggle dataset and use it in a fresh notebook for predictive modelling.


In [ ]:
# recompute normalised topic proportions for the cleaned rows
topic_cols = [c for c in features_clean.columns if c.startswith("topic_") and "_prop" not in c]

tw = features_clean[topic_cols].values
row_sums = tw.sum(axis=1, keepdims=True)
topic_props_clean = tw / row_sums

prop_cols = [f"{c}_prop" for c in topic_cols]
for j, col in enumerate(prop_cols):
    features_clean[col] = topic_props_clean[:, j]

cols_export = (
    ["pubDate_dt", "year", "title",
     "dominant_topic", "max_topic_prop", "strongly_dominant",
     "n_topics_prop_over_0_1", "n_topics_to_cover_0_7"]
    + prop_cols
)

dataset_for_model = features_clean[cols_export].copy()
csv_path = "/kaggle/working/bbc_topics_features_clean.csv"
dataset_for_model.to_csv(csv_path, index=False)

csv_path, dataset_for_model.head()
